# Thai Election OCR Agentic Pipeline

Pipeline in this notebook:

1. Read each page image directly
2. OCR every page with EasyOCR
3. Save scanned document data to `data/labels/*.json`
4. Convert Thai digits and OCR number noise into Arabic numerals
5. Classify which pages contain usable table rows
6. Merge multi-page documents and map `party -> votes`
7. Export `data/submission.csv` in `id,votes` format

The notebook is split into sections so you can run the scan stage and submission stage separately.


## 1. Setup

This cell makes sure `tqdm` is available for loading bars. `easyocr` is installed only when the scan stage actually runs.


In [1]:
from __future__ import annotations

import importlib.util
import subprocess
import sys


def ensure_package(package: str, import_name: str | None = None) -> None:
    module_name = import_name or package.replace('-', '_')
    if importlib.util.find_spec(module_name) is None:
        print(f'Installing {package} ...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package])


ensure_package('tqdm')
print('Ready: tqdm installed or already available')

Ready: tqdm installed or already available


## 2. Config

Adjust the flags in this section before you run the scan or submission stages.

In [2]:
import csv
import importlib.util
import json
import os
import re
import subprocess
import sys
import time
import unicodedata
from collections import defaultdict
from dataclasses import dataclass
from difflib import SequenceMatcher
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

from PIL import Image
from tqdm.auto import tqdm

DATA_ROOT = Path('data')
IMAGE_DIR = DATA_ROOT / 'images'
SAMPLE_LABEL_DIR = DATA_ROOT / 'sample_labels'
TEMPLATE_PATH = DATA_ROOT / 'submission_template.csv'
if not TEMPLATE_PATH.exists():
    fallback_template_path = DATA_ROOT / 'submission_template copy.csv'
    if fallback_template_path.exists():
        TEMPLATE_PATH = fallback_template_path
SUBMISSION_SCAFFOLD_PATH = DATA_ROOT / 'submission_template_v3.csv'
LABEL_DIR = DATA_ROOT / 'labels'
OCR_CACHE_DIR = DATA_ROOT / 'ocr_cache_easyocr'
OUTPUT_PATH = DATA_ROOT / 'submission.csv'
PARTIAL_OUTPUT_PATH = DATA_ROOT / 'submission.partial.csv'
ENV_PATH = Path('.env')

EASYOCR_LANGUAGES = ['th', 'en']
EASYOCR_USE_GPU = True
EASYOCR_CONFIDENCE_THRESHOLD = 0.10
EASYOCR_MAG_RATIO = 1.2
EASYOCR_CANVAS_SIZE = 2560
MIN_INTERVAL_SECONDS = 0.0
PAUSE_EVERY_PAGES = 0
PAUSE_SECONDS = 0
MAX_RETRIES = 2
RETRY_BASE_SECONDS = 5
TABLE_MIN_REGION_AREA_RATIO = 0.0025
TABLE_MIN_WIDTH_RATIO = 0.18
TABLE_MIN_HEIGHT_RATIO = 0.035
TABLE_MIN_LINE_DENSITY = 0.015
TABLE_FALLBACK_DENSITY = 0.0025
TABLE_DETECTION_MAX_SIDE = 1800
TABLE_REGION_PADDING = 16
RESCAN_EXISTING_LABELS = False
FORCE_RERUN_OCR = False
STORE_MARKDOWN_IN_LABELS = True
ALLOW_MISSING_LABELS = False
RUN_SCAN_TO_LABELS = False
RUN_BUILD_SUBMISSION = True
LIMIT_DOCS: Optional[int] = None

assert IMAGE_DIR.exists(), f'Missing image directory: {IMAGE_DIR}'
assert TEMPLATE_PATH.exists(), f'Missing template file: {TEMPLATE_PATH}'
assert SUBMISSION_SCAFFOLD_PATH.exists(), f'Missing submission scaffold file: {SUBMISSION_SCAFFOLD_PATH}'
LABEL_DIR.mkdir(parents=True, exist_ok=True)
OCR_CACHE_DIR.mkdir(parents=True, exist_ok=True)


def load_dotenv_file(path: Path = ENV_PATH) -> None:
    if not path.exists():
        return
    for raw_line in path.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        if key:
            os.environ.setdefault(key, value)


load_dotenv_file()
print(f'EasyOCR backend selected | languages={EASYOCR_LANGUAGES} | gpu={EASYOCR_USE_GPU}')
print(f'RUN_SCAN_TO_LABELS={RUN_SCAN_TO_LABELS}, RUN_BUILD_SUBMISSION={RUN_BUILD_SUBMISSION}, LIMIT_DOCS={LIMIT_DOCS}')


@dataclass(frozen=True)
class TemplateRow:
    id: str
    doc_id: str
    row_num: int
    party_name: str


@dataclass
class ExtractedRow:
    party_name: str
    votes_text: str
    source_page: Optional[int] = None
    candidate_name: Optional[str] = None
    rank_hint: Optional[int] = None
    raw_line: Optional[str] = None
    match_score: Optional[float] = None


EasyOCR backend selected | languages=['th', 'en'] | gpu=True
RUN_SCAN_TO_LABELS=False, RUN_BUILD_SUBMISSION=True, LIMIT_DOCS=None


c:\Users\skizz\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. Load Competition Structure

This section loads the document index, the template party mapping, and province metadata from the sample labels.

In [ ]:
PAGE_RE = re.compile(r'^(?P<doc_id>.+?)(?:_page(?P<page>\d+))?\.png$')


def parse_page_record(path: Path) -> Tuple[str, int]:
    match = PAGE_RE.match(path.name)
    if not match:
        raise ValueError(f'Unexpected image name: {path.name}')
    doc_id = match.group('doc_id')
    page_number = int(match.group('page') or '1')
    return doc_id, page_number


def build_document_index(image_dir: Path) -> Dict[str, List[Path]]:
    grouped: Dict[str, List[Tuple[int, Path]]] = defaultdict(list)
    for path in sorted(image_dir.glob('*.png')):
        doc_id, page_number = parse_page_record(path)
        grouped[doc_id].append((page_number, path))
    return {doc_id: [path for page_number, path in sorted(items)] for doc_id, items in grouped.items()}


def load_template(path: Path) -> List[TemplateRow]:
    rows: List[TemplateRow] = []
    with path.open('r', encoding='utf-8', newline='') as f:
        reader = csv.DictReader(f)
        for row in reader:
            rows.append(
                TemplateRow(
                    id=row['id'],
                    doc_id=row['doc_id'],
                    row_num=int(row['row_num']),
                    party_name=row['party_name'],
                )
            )
    return rows


def group_template_by_doc(rows: Sequence[TemplateRow]) -> Dict[str, List[TemplateRow]]:
    grouped: Dict[str, List[TemplateRow]] = defaultdict(list)
    for row in rows:
        grouped[row.doc_id].append(row)
    for doc_id in grouped:
        grouped[doc_id].sort(key=lambda row: row.row_num)
    return dict(grouped)


def parse_doc_id(doc_id: str) -> Dict[str, Any]:
    parts = doc_id.split('_')
    if parts[:2] == ['party', 'list'] and len(parts) >= 4:
        return {'type': 'party_list', 'province_code': parts[2], 'constituency_number': int(parts[3])}
    if parts[0] == 'constituency' and len(parts) >= 3:
        return {'type': 'constituency', 'province_code': parts[1], 'constituency_number': int(parts[2])}
    raise ValueError(f'Unexpected doc_id format: {doc_id}')


def build_known_province_names(sample_dir: Path) -> Dict[str, Optional[str]]:
    province_names: Dict[str, Optional[str]] = {}
    for path in sorted(sample_dir.glob('*.json')):
        payload = json.loads(path.read_text(encoding='utf-8'))
        province_code = str(payload.get('province_code')) if payload.get('province_code') is not None else None
        if province_code:
            province_names[province_code] = payload.get('province_name')
    return province_names


document_index = build_document_index(IMAGE_DIR)
template_rows = load_template(TEMPLATE_PATH)
template_by_doc = group_template_by_doc(template_rows)
known_province_names = build_known_province_names(SAMPLE_LABEL_DIR)

doc_ids = sorted(template_by_doc)
if LIMIT_DOCS:
    doc_ids = doc_ids[:LIMIT_DOCS]

print(f'Documents in images: {len(document_index)}')
print(f'Documents in template: {len(template_by_doc)}')
print(f'Template rows: {len(template_rows)}')
print(f'Known province names from sample labels: {len(known_province_names)}')
print(f'Active documents for this run: {len(doc_ids)}')

## 4. Text Normalization, Page Classification, and Mapping

This section handles Thai-digit normalization, OCR cleanup, page classification, party matching, and final `id,votes` writing.

In [ ]:
THAI_DIGITS = str.maketrans('๐๑๒๓๔๕๖๗๘๙', '0123456789')
COMMON_DIGIT_CONFUSIONS = {
    'O': '0', 'o': '0', 'Q': '0', 'D': '0', 'U': '0',
    'I': '1', 'l': '1', '|': '1', '!': '1',
    'Z': '2',
    'S': '5', 's': '5',
    'B': '8',
    ',': '', '.': '', ' ': '', '-': '', '_': '', '/': '',
}
SUMMARY_KEYWORDS = [
    '????????????????????????',
    '??????????????????',
    '???????????????????',
    '??????',
    '????????',
    '?????????????????????????',
    '?????????????',
]


def normalize_text(text: Optional[str]) -> str:
    if text is None:
        return ''
    text = unicodedata.normalize('NFKC', str(text))
    text = text.replace('​', ' ').replace(' ', ' ')
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def normalize_markdown_text(text: Optional[str]) -> str:
    if text is None:
        return ''
    text = unicodedata.normalize('NFKC', str(text)).replace('\r\n', '\n').replace('\r', '\n')
    normalized_lines = [line.rstrip() for line in text.split('\n')]
    return '\n'.join(normalized_lines).strip()




def normalize_party_key(text: Optional[str]) -> str:
    text = normalize_text(text).replace(' ', '')
    text = re.sub(r'[^\w฀-๿]', '', text)
    return text.casefold()


def normalize_vote_text(text: Optional[str]) -> str:
    text = normalize_text(text).translate(THAI_DIGITS)
    for src, dst in COMMON_DIGIT_CONFUSIONS.items():
        text = text.replace(src, dst)
    text = re.sub(r'[^0-9]', '', text)
    return text


def coerce_votes(text: Optional[str], default: str = '0') -> str:
    digits = normalize_vote_text(text)
    return digits if digits else default


def similarity(a: str, b: str) -> float:
    return SequenceMatcher(None, normalize_party_key(a), normalize_party_key(b)).ratio()


def best_party_match(observed_party: str, allowed_parties: Sequence[str], threshold: float = 0.55) -> Tuple[Optional[str], float]:
    if not allowed_parties:
        return None, 0.0
    scored = sorted(((party, similarity(observed_party, party)) for party in allowed_parties), key=lambda item: item[1], reverse=True)
    best_party, best_score = scored[0]
    if best_score < threshold:
        return None, best_score
    return best_party, best_score


def choose_best_vote(existing: Optional[str], incoming: Optional[str]) -> str:
    existing_digits = normalize_vote_text(existing)
    incoming_digits = normalize_vote_text(incoming)
    if incoming_digits and not existing_digits:
        return incoming_digits
    if existing_digits and not incoming_digits:
        return existing_digits
    if len(incoming_digits) > len(existing_digits):
        return incoming_digits
    return existing_digits or incoming_digits or '0'


def template_lookup(rows: Sequence[TemplateRow]) -> Dict[str, TemplateRow]:
    return {normalize_party_key(row.party_name): row for row in rows}


def iter_markdown_lines(markdown_text: str) -> Iterable[str]:
    for raw_line in normalize_markdown_text(markdown_text).splitlines():
        line = normalize_text(raw_line.strip())
        if not line:
            continue
        if re.fullmatch(r'\|?\s*:?-{3,}:?\s*(\|\s*:?-{3,}:?\s*)+\|?', line):
            continue
        line = line.strip('|')
        parts = [normalize_text(part) for part in line.split('|') if normalize_text(part)]
        if parts:
            yield ' | '.join(parts)
        else:
            yield line


def extract_vote_candidate(text: str) -> str:
    matches = re.findall(r'[0-9๐-๙OolIQDSB,._\- ]{1,20}', text)
    best = ''
    for match in matches:
        digits = normalize_vote_text(match)
        if len(digits) >= len(best):
            best = digits
    return best


def extract_rank_hint(text: str) -> Optional[int]:
    match = re.match(r'^\D{0,4}(\d{1,3})\D', normalize_text(text))
    if not match:
        return None
    return int(match.group(1))


def extract_candidate_name(text: str, matched_party: Optional[str], votes_text: str) -> Optional[str]:
    candidate = normalize_text(text)
    if matched_party:
        candidate = candidate.replace(matched_party, ' ')
    if votes_text:
        candidate = candidate.replace(votes_text, ' ')
    candidate = re.sub(r'\b\d{1,3}\b', ' ', candidate)
    candidate = normalize_text(candidate.replace('|', ' '))
    return candidate or None


def serialize_extracted_row(row: ExtractedRow) -> Dict[str, Any]:
    return {
        'party': row.party_name,
        'votes': int(coerce_votes(row.votes_text)),
        'source_page': row.source_page,
        'candidate_name': row.candidate_name,
        'rank_hint': row.rank_hint,
        'raw_line': row.raw_line,
        'match_score': row.match_score,
    }


def deserialize_extracted_row(item: Dict[str, Any]) -> ExtractedRow:
    return ExtractedRow(
        party_name=str(item.get('party', '')),
        votes_text=str(item.get('votes', '0')),
        source_page=item.get('source_page'),
        candidate_name=item.get('candidate_name'),
        rank_hint=item.get('rank_hint'),
        raw_line=item.get('raw_line'),
        match_score=item.get('match_score'),
    )


def parse_markdown_rows(markdown_text: str, template_rows_for_doc: Sequence[TemplateRow], source_page: int) -> List[ExtractedRow]:
    allowed_parties = [row.party_name for row in template_rows_for_doc]
    parsed_rows: List[ExtractedRow] = []
    for line in iter_markdown_lines(markdown_text):
        segments = [segment for segment in line.split(' | ') if normalize_text(segment)]
        if not segments:
            continue
        if len(segments) == 1 and any(keyword in line for keyword in SUMMARY_KEYWORDS):
            continue
        vote_candidates = [extract_vote_candidate(segment) for segment in reversed(segments)]
        vote_candidates.append(extract_vote_candidate(line))
        votes = max(vote_candidates, key=len, default='')
        if not votes:
            continue
        matched_party = None
        matched_score = 0.0
        for target in segments + [line]:
            candidate_party, score = best_party_match(target, allowed_parties, threshold=0.56)
            if candidate_party is not None and score > matched_score:
                matched_party = candidate_party
                matched_score = score
        if matched_party is None:
            continue
        parsed_rows.append(
            ExtractedRow(
                party_name=matched_party,
                votes_text=votes,
                source_page=source_page,
                candidate_name=extract_candidate_name(line, matched_party, votes),
                rank_hint=extract_rank_hint(line),
                raw_line=line,
                match_score=round(matched_score, 4),
            )
        )
    return parsed_rows


def classify_page_content(
    markdown_text: str,
    template_rows_for_doc: Sequence[TemplateRow],
    page_number: int,
    parsed_rows: Optional[Sequence[ExtractedRow]] = None,
    image_table_score: float = 0.0,
    detected_table_regions: Optional[Sequence[Sequence[int]]] = None,
) -> Dict[str, Any]:
    rows = list(parsed_rows or [])
    table_region_count = len(detected_table_regions or [])
    unique_party_hits = len({normalize_party_key(row.party_name) for row in rows if normalize_party_key(row.party_name)})
    row_hits = len(rows)
    numeric_hits = sum(1 for row in rows if normalize_vote_text(row.votes_text))
    summary_hits = sum(1 for keyword in SUMMARY_KEYWORDS if keyword in normalize_markdown_text(markdown_text))
    has_table_bars = '|' in markdown_text
    signal_score = min(
        1.0,
        (image_table_score * 0.55) +
        (table_region_count * 0.10) +
        (unique_party_hits * 0.16) +
        (row_hits * 0.12) +
        (numeric_hits * 0.06) +
        (0.08 if has_table_bars else 0.0) +
        (0.06 if page_number in {1, 2, 3, 4} else 0.0),
    )
    signal_score = round(signal_score, 4)
    has_image_table = table_region_count > 0 or image_table_score >= 0.22
    if has_image_table and (row_hits >= 1 or unique_party_hits >= 1 or numeric_hits >= 1):
        page_type = 'table'
    elif has_image_table:
        page_type = 'table_candidate'
    elif row_hits >= 2 or (unique_party_hits >= 1 and numeric_hits >= 2):
        page_type = 'mixed'
    elif page_number == 1:
        page_type = 'header'
    elif summary_hits >= 2:
        page_type = 'summary'
    else:
        page_type = 'other'
    use_for_results = page_type in {'table', 'mixed'} and row_hits > 0
    return {
        'page_type': page_type,
        'signal_score': signal_score,
        'party_hits': unique_party_hits,
        'row_hits': row_hits,
        'numeric_hits': numeric_hits,
        'summary_hits': summary_hits,
        'table_region_count': table_region_count,
        'image_table_score': round(image_table_score, 4),
        'use_for_results': use_for_results,
    }


def map_rows_to_template(doc_id: str, extracted_rows: Sequence[ExtractedRow], template_rows_for_doc: Sequence[TemplateRow]) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    by_party = template_lookup(template_rows_for_doc)
    allowed_parties = [row.party_name for row in template_rows_for_doc]
    votes_by_party: Dict[str, str] = {row.party_name: '0' for row in template_rows_for_doc}
    diagnostics: List[Dict[str, Any]] = []
    for extracted in extracted_rows:
        raw_party = normalize_text(extracted.party_name)
        if not raw_party:
            diagnostics.append({'doc_id': doc_id, 'issue': 'missing_party_name', 'row': serialize_extracted_row(extracted)})
            continue
        direct = by_party.get(normalize_party_key(raw_party))
        if direct is not None:
            matched_party = direct.party_name
            score = 1.0
        else:
            matched_party, score = best_party_match(raw_party, allowed_parties)
            if matched_party is None:
                diagnostics.append({'doc_id': doc_id, 'issue': 'unmatched_party_name', 'score': score, 'row': serialize_extracted_row(extracted)})
                continue
        votes_by_party[matched_party] = choose_best_vote(votes_by_party.get(matched_party), extracted.votes_text)
        diagnostics.append({
            'doc_id': doc_id,
            'issue': 'mapped',
            'observed_party': raw_party,
            'matched_party': matched_party,
            'score': round(score, 4),
            'votes': votes_by_party[matched_party],
            'source_page': extracted.source_page,
        })
    submission_rows: List[Dict[str, Any]] = []
    for row in template_rows_for_doc:
        submission_rows.append({
            'id': row.id,
            'doc_id': row.doc_id,
            'row_num': row.row_num,
            'party_name': row.party_name,
            'votes': coerce_votes(votes_by_party.get(row.party_name, '0')),
        })
    return submission_rows, diagnostics


def build_result_items(doc_id: str, submission_rows: Sequence[Dict[str, Any]], extracted_rows: Sequence[ExtractedRow]) -> List[Dict[str, Any]]:
    doc_meta = parse_doc_id(doc_id)
    best_candidate_name: Dict[str, str] = {}
    best_rank_hint: Dict[str, int] = {}
    for row in extracted_rows:
        key = normalize_party_key(row.party_name)
        if row.candidate_name and len(row.candidate_name) > len(best_candidate_name.get(key, '')):
            best_candidate_name[key] = row.candidate_name
        if row.rank_hint is not None and key not in best_rank_hint:
            best_rank_hint[key] = row.rank_hint
    results: List[Dict[str, Any]] = []
    for row in submission_rows:
        key = normalize_party_key(row['party_name'])
        item: Dict[str, Any] = {
            'number': best_rank_hint.get(key, int(row['row_num'])),
            'party': row['party_name'],
            'votes': int(coerce_votes(row.get('votes', '0'))),
        }
        if doc_meta['type'] == 'constituency':
            item['name'] = best_candidate_name.get(key)
        results.append(item)
    return results


def write_submission(rows: Sequence[Dict[str, Any]], output_path: Path) -> None:
    fieldnames = ['id', 'votes']
    with output_path.open('w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow({'id': row['id'], 'votes': coerce_votes(row.get('votes', '0'))})


## 5. EasyOCR One Page at a Time

This stage OCRs one page at a time with EasyOCR, writes one OCR cache file per page, and keeps the same page-label storage flow as the previous notebook.


In [ ]:
LAST_EASYOCR_CALL_AT = 0.0
OCR_REQUEST_COUNT = 0
_EASYOCR_READER = None
_CV2 = None
_NP = None


def get_page_cache_path(image_path: Path) -> Path:
    return OCR_CACHE_DIR / f'{image_path.stem}.md'


def ensure_vision_libs():
    global _CV2, _NP
    if _CV2 is not None and _NP is not None:
        return _CV2, _NP
    try:
        import cv2 as cv2_module
    except ImportError:
        print('Installing opencv-python-headless ...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'opencv-python-headless'])
        import cv2 as cv2_module
    try:
        import numpy as np_module
    except ImportError:
        print('Installing numpy ...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'numpy'])
        import numpy as np_module
    _CV2 = cv2_module
    _NP = np_module
    return _CV2, _NP


def ensure_easyocr_reader():
    global _EASYOCR_READER
    if _EASYOCR_READER is not None:
        return _EASYOCR_READER
    try:
        import easyocr
    except ImportError:
        print('Installing easyocr ...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'easyocr'])
        import easyocr
    _EASYOCR_READER = easyocr.Reader(EASYOCR_LANGUAGES, gpu=EASYOCR_USE_GPU)
    return _EASYOCR_READER


def is_retryable_easyocr_error(exc: Exception) -> bool:
    message = str(exc).lower()
    keywords = ['timeout', 'temporarily', 'connection', 'cuda', 'memory', 'resource']
    return any(keyword in message for keyword in keywords)


def before_ocr_request() -> None:
    global LAST_EASYOCR_CALL_AT, OCR_REQUEST_COUNT
    if PAUSE_EVERY_PAGES > 0 and PAUSE_SECONDS > 0 and OCR_REQUEST_COUNT > 0 and OCR_REQUEST_COUNT % PAUSE_EVERY_PAGES == 0:
        print(f'Cooling down for {PAUSE_SECONDS} seconds after {OCR_REQUEST_COUNT} EasyOCR pages ...')
        time.sleep(PAUSE_SECONDS)
    wait_seconds = MIN_INTERVAL_SECONDS - (time.time() - LAST_EASYOCR_CALL_AT)
    if wait_seconds > 0:
        time.sleep(wait_seconds)

def clear_torch_cuda_cache() -> None:
    try:
        import gc
        import torch
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass


def readtext_with_fallbacks(reader: Any, image: Any) -> Sequence[Any]:
    settings = [
        {'canvas_size': EASYOCR_CANVAS_SIZE, 'mag_ratio': EASYOCR_MAG_RATIO},
        {'canvas_size': 1920, 'mag_ratio': 1.0},
        {'canvas_size': 1600, 'mag_ratio': 1.0},
    ]
    last_exc: Optional[Exception] = None
    for opts in settings:
        try:
            clear_torch_cuda_cache()
            return reader.readtext(
                image,
                detail=1,
                paragraph=False,
                contrast_ths=0.05,
                adjust_contrast=0.7,
                text_threshold=0.45,
                low_text=0.2,
                link_threshold=0.25,
                canvas_size=opts['canvas_size'],
                mag_ratio=opts['mag_ratio'],
            )
        except Exception as exc:
            last_exc = exc
            if 'out of memory' not in str(exc).lower():
                raise
    raise RuntimeError(f'EasyOCR region failed after memory fallbacks: {last_exc}') from last_exc


def _boxes_touch(box_a: Tuple[int, int, int, int], box_b: Tuple[int, int, int, int], gap: int) -> bool:
    ax, ay, aw, ah = box_a
    bx, by, bw, bh = box_b
    return not (
        (ax + aw + gap) < bx or
        (bx + bw + gap) < ax or
        (ay + ah + gap) < by or
        (by + bh + gap) < ay
    )


def _merge_boxes(boxes: Sequence[Tuple[int, int, int, int]], gap: int) -> List[Tuple[int, int, int, int]]:
    pending = [tuple(box) for box in boxes]
    merged: List[Tuple[int, int, int, int]] = []
    while pending:
        x, y, w, h = pending.pop(0)
        changed = True
        while changed:
            changed = False
            remaining: List[Tuple[int, int, int, int]] = []
            for other in pending:
                if _boxes_touch((x, y, w, h), other, gap):
                    ox, oy, ow, oh = other
                    x2 = max(x + w, ox + ow)
                    y2 = max(y + h, oy + oh)
                    x = min(x, ox)
                    y = min(y, oy)
                    w = x2 - x
                    h = y2 - y
                    changed = True
                else:
                    remaining.append(other)
            pending = remaining
        merged.append((x, y, w, h))
    return sorted(merged, key=lambda item: (item[1], item[0]))


def _expand_box(box: Tuple[int, int, int, int], image_shape: Tuple[int, int], padding: int) -> Tuple[int, int, int, int]:
    height, width = image_shape
    x, y, w, h = box
    x1 = max(0, x - padding)
    y1 = max(0, y - padding)
    x2 = min(width, x + w + padding)
    y2 = min(height, y + h + padding)
    return x1, y1, x2 - x1, y2 - y1


def detect_table_regions(image_path: Path, page_number: Optional[int] = None) -> Tuple[List[List[int]], Dict[str, Any]]:
    cv2, _ = ensure_vision_libs()
    gray = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    if gray is None:
        raise FileNotFoundError(f'Unable to load image: {image_path}')
    orig_h, orig_w = gray.shape[:2]
    resize_ratio = 1.0
    if max(orig_h, orig_w) > TABLE_DETECTION_MAX_SIDE:
        resize_ratio = TABLE_DETECTION_MAX_SIDE / float(max(orig_h, orig_w))
        working = cv2.resize(gray, (max(1, int(orig_w * resize_ratio)), max(1, int(orig_h * resize_ratio))), interpolation=cv2.INTER_AREA)
    else:
        working = gray
    work_h, work_w = working.shape[:2]
    inverted = cv2.bitwise_not(working)
    binary = cv2.adaptiveThreshold(inverted, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 35, -5)
    horizontal_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (max(20, work_w // 30), 1))
    vertical_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, max(20, work_h // 45)))
    horizontal = cv2.morphologyEx(binary, cv2.MORPH_OPEN, horizontal_kernel, iterations=1)
    vertical = cv2.morphologyEx(binary, cv2.MORPH_OPEN, vertical_kernel, iterations=1)
    line_mask = cv2.add(horizontal, vertical)
    line_mask = cv2.dilate(line_mask, cv2.getStructuringElement(cv2.MORPH_RECT, (7, 7)), iterations=1)
    contours_info = cv2.findContours(line_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    contours = contours_info[0] if len(contours_info) == 2 else contours_info[1]
    candidate_boxes: List[Tuple[int, int, int, int]] = []
    candidate_scores: List[float] = []
    image_area = float(max(1, work_h * work_w))
    for contour in contours:
        x, y, w, h = cv2.boundingRect(contour)
        area = float(max(1, w * h))
        area_ratio = area / image_area
        width_ratio = w / float(max(1, work_w))
        height_ratio = h / float(max(1, work_h))
        region = line_mask[y:y + h, x:x + w]
        density = cv2.countNonZero(region) / area
        strong_wide = width_ratio >= TABLE_MIN_WIDTH_RATIO and height_ratio >= TABLE_MIN_HEIGHT_RATIO
        compact_bottom = (y / float(max(1, work_h)) >= 0.40) and width_ratio >= (TABLE_MIN_WIDTH_RATIO * 0.7) and height_ratio >= (TABLE_MIN_HEIGHT_RATIO * 0.7)
        if area_ratio < TABLE_MIN_REGION_AREA_RATIO:
            continue
        if density < TABLE_MIN_LINE_DENSITY and not (strong_wide or compact_bottom):
            continue
        if not strong_wide and not compact_bottom:
            continue
        candidate_boxes.append((x, y, w, h))
        candidate_scores.append(min(1.0, area_ratio * 10.0 + density * 2.0))
    fallback_used = False
    if not candidate_boxes and page_number in {1, 2, 3, 4}:
        band_start = int(work_h * 0.45)
        band = line_mask[band_start:, :]
        band_density = cv2.countNonZero(band) / float(max(1, band.size))
        non_zero = cv2.findNonZero(band)
        if non_zero is not None and band_density >= TABLE_FALLBACK_DENSITY:
            x, y, w, h = cv2.boundingRect(non_zero)
            y += band_start
            width_ratio = w / float(max(1, work_w))
            height_ratio = h / float(max(1, work_h))
            if width_ratio >= 0.20 and height_ratio >= 0.03:
                candidate_boxes.append((x, y, w, h))
                candidate_scores.append(min(1.0, band_density * 40.0 + width_ratio * 0.3 + height_ratio * 0.8))
                fallback_used = True
    merge_gap = max(12, int(min(work_h, work_w) * 0.02))
    merged_boxes = _merge_boxes(candidate_boxes, gap=merge_gap)
    scaled_boxes: List[List[int]] = []
    for x, y, w, h in merged_boxes:
        if resize_ratio != 1.0:
            x = int(round(x / resize_ratio))
            y = int(round(y / resize_ratio))
            w = int(round(w / resize_ratio))
            h = int(round(h / resize_ratio))
        ex, ey, ew, eh = _expand_box((x, y, w, h), (orig_h, orig_w), TABLE_REGION_PADDING)
        scaled_boxes.append([int(ex), int(ey), int(ew), int(eh)])
    best_score = max(candidate_scores, default=0.0)
    diagnostics = {
        'table_score': round(min(1.0, best_score + (0.12 * len(scaled_boxes))), 4),
        'fallback_used': fallback_used,
        'raw_region_count': len(candidate_boxes),
        'merged_region_count': len(scaled_boxes),
    }
    return scaled_boxes, diagnostics


def _line_sort_key(item: Tuple[float, float, float, float, str]) -> Tuple[float, float]:
    return (item[0], item[1])


def easyocr_results_to_markdown(results: Sequence[Any]) -> str:
    line_items: List[Tuple[float, float, float, float, str]] = []
    heights: List[float] = []
    for entry in results:
        if not isinstance(entry, (list, tuple)) or len(entry) < 2:
            continue
        bbox = entry[0]
        text = normalize_text(entry[1])
        confidence = float(entry[2]) if len(entry) >= 3 else 1.0
        if not text:
            continue
        if confidence < EASYOCR_CONFIDENCE_THRESHOLD and len(normalize_vote_text(text)) < 2:
            continue
        xs = [float(point[0]) for point in bbox]
        ys = [float(point[1]) for point in bbox]
        left = min(xs)
        right = max(xs)
        center_y = sum(ys) / len(ys)
        height = max(ys) - min(ys)
        heights.append(height)
        line_items.append((center_y, left, right, height, text))
    if not line_items:
        return ''
    line_items.sort(key=_line_sort_key)
    median_height = sorted(heights)[len(heights) // 2] if heights else 18.0
    y_threshold = max(10.0, median_height * 0.7)
    grouped_lines: List[List[Tuple[float, float, float, float, str]]] = []
    line_centers: List[float] = []
    for item in line_items:
        center_y = item[0]
        if not grouped_lines or abs(center_y - line_centers[-1]) > y_threshold:
            grouped_lines.append([item])
            line_centers.append(center_y)
        else:
            grouped_lines[-1].append(item)
            line_centers[-1] = (line_centers[-1] + center_y) / 2.0
    markdown_lines: List[str] = []
    gap_threshold = max(18.0, median_height * 1.2)
    for line in grouped_lines:
        ordered = sorted(line, key=lambda item: item[1])
        line_text = ordered[0][4]
        prev_right = ordered[0][2]
        for item in ordered[1:]:
            separator = ' | ' if (item[1] - prev_right) > gap_threshold else ' '
            line_text += separator + item[4]
            prev_right = item[2]
        markdown_lines.append(line_text)
    return normalize_markdown_text('\n'.join(markdown_lines))


def _prepare_crop_for_ocr(gray_crop: Any) -> Any:
    cv2, _ = ensure_vision_libs()
    if gray_crop is None or getattr(gray_crop, 'size', 0) == 0:
        return gray_crop
    crop = gray_crop
    height, width = crop.shape[:2]
    if min(height, width) < 700:
        scale = min(1.8, max(1.15, 720.0 / float(max(1, min(height, width)))))
        crop = cv2.resize(crop, (max(1, int(width * scale)), max(1, int(height * scale))), interpolation=cv2.INTER_CUBIC)
    crop = cv2.GaussianBlur(crop, (3, 3), 0)
    return crop


def ocr_image_to_markdown(image_path: Path, table_regions: Optional[Sequence[Sequence[int]]] = None) -> Tuple[str, Path]:
    global LAST_EASYOCR_CALL_AT, OCR_REQUEST_COUNT
    cv2, _ = ensure_vision_libs()
    cache_path = get_page_cache_path(image_path)
    if cache_path.exists() and not FORCE_RERUN_OCR:
        return normalize_markdown_text(cache_path.read_text(encoding='utf-8')), cache_path
    if not table_regions:
        cache_path.write_text('', encoding='utf-8')
        return '', cache_path
    reader = ensure_easyocr_reader()
    gray = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    if gray is None:
        raise FileNotFoundError(f'Unable to load image: {image_path}')
    last_exc: Optional[Exception] = None
    doc_id, page_number = parse_page_record(image_path)
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            before_ocr_request()
            OCR_REQUEST_COUNT += 1
            LAST_EASYOCR_CALL_AT = time.time()
            markdown_blocks: List[str] = []
            for box in table_regions:
                x, y, w, h = [int(value) for value in box]
                crop = gray[y:y + h, x:x + w]
                prepared = _prepare_crop_for_ocr(crop)
                results = readtext_with_fallbacks(reader, prepared)
                block_markdown = easyocr_results_to_markdown(results)
                if block_markdown:
                    markdown_blocks.append(block_markdown)
            markdown = normalize_markdown_text('\n\n'.join(markdown_blocks))
            cache_path.write_text(markdown, encoding='utf-8')
            return markdown, cache_path
        except Exception as exc:
            last_exc = exc
            if attempt >= MAX_RETRIES or not is_retryable_easyocr_error(exc):
                raise
            sleep_seconds = RETRY_BASE_SECONDS * (2 ** (attempt - 1))
            print(f'Retryable EasyOCR error on {doc_id} page {page_number}: {exc}. Sleeping {sleep_seconds}s.')
            time.sleep(sleep_seconds)
    raise RuntimeError(f'EasyOCR failed for {image_path.name}') from last_exc


def scan_page(image_path: Path, template_rows_for_doc: Sequence[TemplateRow]) -> Dict[str, Any]:
    doc_id, page_number = parse_page_record(image_path)
    table_regions, table_region_diagnostics = detect_table_regions(image_path, page_number=page_number)
    markdown_text, cache_path = ocr_image_to_markdown(image_path, table_regions=table_regions)
    parsed_rows = parse_markdown_rows(markdown_text, template_rows_for_doc, source_page=page_number) if table_regions else []
    classification = classify_page_content(
        markdown_text,
        template_rows_for_doc,
        page_number,
        parsed_rows=parsed_rows,
        image_table_score=table_region_diagnostics['table_score'],
        detected_table_regions=table_regions,
    )
    return {
        'doc_id': doc_id,
        'page_number': page_number,
        'source_image_path': str(image_path),
        'page_type': classification['page_type'],
        'signal_score': classification['signal_score'],
        'party_hits': classification['party_hits'],
        'row_hits': classification['row_hits'],
        'numeric_hits': classification['numeric_hits'],
        'summary_hits': classification['summary_hits'],
        'use_for_results': classification['use_for_results'],
        'image_table_score': classification['image_table_score'],
        'table_regions': table_regions,
        'ocr_region_count': len(table_regions),
        'ocr_scope': 'table_regions' if table_regions else 'blank_no_table',
        'table_region_diagnostics': table_region_diagnostics,
        'ocr_cache_path': str(cache_path),
        'ocr_markdown': markdown_text if STORE_MARKDOWN_IN_LABELS else None,
        'parsed_rows': [serialize_extracted_row(row) for row in parsed_rows],
    }


## 6. Scan Every Sheet and Save One JSON per Sheet

This stage OCRs each image separately and writes one JSON file per sheet into `data/labels`.

In [ ]:
def get_page_label_path(image_path: Path) -> Path:
    return LABEL_DIR / f'{image_path.stem}.json'


def build_page_results(doc_id: str, extracted_rows: Sequence[ExtractedRow]) -> List[Dict[str, Any]]:
    doc_meta = parse_doc_id(doc_id)
    grouped: Dict[str, ExtractedRow] = {}
    for row in extracted_rows:
        key = normalize_party_key(row.party_name)
        existing = grouped.get(key)
        if existing is None:
            grouped[key] = row
            continue
        grouped[key] = ExtractedRow(
            party_name=existing.party_name,
            votes_text=choose_best_vote(existing.votes_text, row.votes_text),
            source_page=existing.source_page,
            candidate_name=existing.candidate_name or row.candidate_name,
            rank_hint=existing.rank_hint if existing.rank_hint is not None else row.rank_hint,
            raw_line=existing.raw_line or row.raw_line,
            match_score=max(existing.match_score or 0.0, row.match_score or 0.0),
        )
    ordered_rows = sorted(grouped.values(), key=lambda row: (row.rank_hint is None, row.rank_hint if row.rank_hint is not None else 9999, row.party_name))
    results: List[Dict[str, Any]] = []
    for idx, row in enumerate(ordered_rows, 1):
        item: Dict[str, Any] = {
            'number': row.rank_hint if row.rank_hint is not None else idx,
            'party': row.party_name,
            'votes': int(coerce_votes(row.votes_text)),
        }
        if doc_meta['type'] == 'constituency':
            item['name'] = row.candidate_name
        results.append(item)
    return results


def build_page_label_payload(image_path: Path, page_info: Dict[str, Any]) -> Dict[str, Any]:
    doc_id, page_number = parse_page_record(image_path)
    doc_meta = parse_doc_id(doc_id)
    extracted_rows = [deserialize_extracted_row(item) for item in page_info.get('parsed_rows', [])]
    payload: Dict[str, Any] = {
        'province_name': known_province_names.get(doc_meta['province_code']),
        'constituency_number': doc_meta['constituency_number'],
        'type': doc_meta['type'],
        'summary_by_section': {},
        'results': build_page_results(doc_id, extracted_rows) if page_info.get('use_for_results') else [],
        'province_code': doc_meta['province_code'],
        'doc_id': doc_id,
        'page_number': page_number,
        'page_type': page_info['page_type'],
        'use_for_results': page_info['use_for_results'],
        'source_image_path': page_info['source_image_path'],
        'ocr_cache_path': page_info['ocr_cache_path'],
        'ocr_markdown': page_info['ocr_markdown'],
        'diagnostics': {
            'signal_score': page_info['signal_score'],
            'party_hits': page_info['party_hits'],
            'row_hits': page_info['row_hits'],
            'numeric_hits': page_info['numeric_hits'],
            'summary_hits': page_info.get('summary_hits', 0),
            'image_table_score': page_info.get('image_table_score', 0.0),
            'ocr_scope': page_info.get('ocr_scope', 'table_regions'),
            'ocr_region_count': page_info.get('ocr_region_count', 0),
            'detected_table_regions': page_info.get('table_regions', []),
            'table_region_diagnostics': page_info.get('table_region_diagnostics', {}),
        },
    }
    return payload


def save_page_label(image_path: Path, payload: Dict[str, Any]) -> Path:
    output_path = get_page_label_path(image_path)
    output_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
    return output_path


def load_page_label(image_path: Path) -> Dict[str, Any]:
    return json.loads(get_page_label_path(image_path).read_text(encoding='utf-8'))


def get_active_image_paths(limit_docs: Optional[int] = LIMIT_DOCS) -> List[Path]:
    active_docs = doc_ids if limit_docs is None else doc_ids[:limit_docs]
    paths: List[Path] = []
    for doc_id in active_docs:
        paths.extend(document_index[doc_id])
    return paths


def scan_image_to_label(image_path: Path) -> Dict[str, Any]:
    label_path = get_page_label_path(image_path)
    if label_path.exists() and not RESCAN_EXISTING_LABELS:
        return load_page_label(image_path)
    doc_id, _ = parse_page_record(image_path)
    page_info = scan_page(image_path, template_by_doc[doc_id])
    payload = build_page_label_payload(image_path, page_info)
    save_page_label(image_path, payload)
    return payload


def scan_all_pages(limit_docs: Optional[int] = LIMIT_DOCS) -> None:
    image_paths = get_active_image_paths(limit_docs=limit_docs)
    for image_path in tqdm(image_paths, desc='scan pages'):
        scan_image_to_label(image_path)
    print(f'Scan complete: {len(image_paths)} page labels in {LABEL_DIR}')


if RUN_SCAN_TO_LABELS:
    scan_all_pages()
else:
    print(f'Set RUN_SCAN_TO_LABELS = True to OCR every sheet and save one JSON per sheet in {LABEL_DIR}')


## 7. Build `submission.csv` from Page Labels

This stage loads one-sheet JSON labels, groups them back to documents, keeps only pages flagged for results, and writes `data/submission.csv` in `id,votes` format.

In [3]:
def label_payload_to_extracted_rows(payload: Dict[str, Any]) -> List[ExtractedRow]:
    rows: List[ExtractedRow] = []
    for item in payload.get('results', []):
        rows.append(
            ExtractedRow(
                party_name=str(item.get('party', '')),
                votes_text=str(item.get('votes', '0')),
                source_page=payload.get('page_number'),
                candidate_name=item.get('name'),
                rank_hint=item.get('number'),
                raw_line=None,
                match_score=None,
            )
        )
    return rows


def load_submission_scaffold(path: Path = SUBMISSION_SCAFFOLD_PATH) -> Tuple[List[Dict[str, str]], List[str]]:
    with path.open('r', encoding='utf-8-sig', newline='') as f:
        reader = csv.DictReader(f)
        fieldnames = list(reader.fieldnames or [])
        if 'id' not in fieldnames or 'votes' not in fieldnames:
            raise ValueError(f'Submission scaffold must contain id and votes columns: {path}')
        rows = [dict(row) for row in reader]
    return rows, fieldnames


def validate_submission_scaffold(scaffold_rows: Sequence[Dict[str, str]], expected_ids: Sequence[str]) -> None:
    if len(scaffold_rows) != len(expected_ids):
        raise ValueError(f'Submission scaffold row count mismatch: expected {len(expected_ids)}, found {len(scaffold_rows)}')
    scaffold_ids = [str(row.get('id', '')).strip() for row in scaffold_rows]
    duplicate_ids = sorted({row_id for row_id in scaffold_ids if row_id and scaffold_ids.count(row_id) > 1})
    if duplicate_ids:
        preview = ', '.join(duplicate_ids[:5])
        raise ValueError(f'Submission scaffold has duplicate ids. Examples: {preview}')
    expected_id_set = set(expected_ids)
    scaffold_id_set = set(scaffold_ids)
    missing_ids = sorted(expected_id_set - scaffold_id_set)
    extra_ids = sorted(scaffold_id_set - expected_id_set)
    if missing_ids or extra_ids:
        raise ValueError(
            f'Submission scaffold id mismatch | missing={len(missing_ids)} extra={len(extra_ids)} '
            f"examples_missing={', '.join(missing_ids[:3])} examples_extra={', '.join(extra_ids[:3])}"
        )


def materialize_submission_rows(votes_by_id: Dict[str, str], scaffold_rows: Sequence[Dict[str, str]]) -> List[Dict[str, str]]:
    output_rows: List[Dict[str, str]] = []
    for row in scaffold_rows:
        row_id = str(row.get('id', '')).strip()
        output_rows.append({'id': row_id, 'votes': coerce_votes(votes_by_id.get(row_id, '0'))})
    return output_rows


def validate_submission_rows(rows: Sequence[Dict[str, str]], expected_ids: Sequence[str]) -> None:
    if len(rows) != len(expected_ids):
        raise ValueError(f'Submission row count mismatch: expected {len(expected_ids)}, found {len(rows)}')
    observed_ids = [str(row.get('id', '')).strip() for row in rows]
    if len(set(observed_ids)) != len(observed_ids):
        raise ValueError('Submission contains duplicate ids')
    missing_ids = sorted(set(expected_ids) - set(observed_ids))
    extra_ids = sorted(set(observed_ids) - set(expected_ids))
    if missing_ids or extra_ids:
        raise ValueError(
            f'Submission id mismatch | missing={len(missing_ids)} extra={len(extra_ids)} '
            f"examples_missing={', '.join(missing_ids[:3])} examples_extra={', '.join(extra_ids[:3])}"
        )
    bad_votes = [row['id'] for row in rows if not re.fullmatch(r'\d+', str(row.get('votes', '')))]
    if bad_votes:
        preview = ', '.join(bad_votes[:5])
        raise ValueError(f'Submission contains non-numeric votes. Examples: {preview}')


def write_submission_rows(rows: Sequence[Dict[str, str]], output_path: Path) -> None:
    with output_path.open('w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['id', 'votes'])
        writer.writeheader()
        for row in rows:
            writer.writerow({'id': row['id'], 'votes': coerce_votes(row.get('votes', '0'))})


def build_submission_from_page_labels(limit_docs: Optional[int] = LIMIT_DOCS) -> List[Dict[str, str]]:
    active_docs = doc_ids if limit_docs is None else doc_ids[:limit_docs]
    expected_ids = [row.id for row in template_rows]
    scaffold_rows, _ = load_submission_scaffold(SUBMISSION_SCAFFOLD_PATH)
    validate_submission_scaffold(scaffold_rows, expected_ids)

    missing_labels: List[str] = []
    votes_by_id: Dict[str, str] = {row_id: '0' for row_id in expected_ids}

    for index, doc_id in enumerate(tqdm(active_docs, desc='build submission'), 1):
        extracted_rows: List[ExtractedRow] = []
        for image_path in document_index[doc_id]:
            label_path = get_page_label_path(image_path)
            if not label_path.exists():
                missing_labels.append(label_path.name)
                continue
            payload = json.loads(label_path.read_text(encoding='utf-8'))
            if not payload.get('use_for_results'):
                continue
            extracted_rows.extend(label_payload_to_extracted_rows(payload))

        submission_rows, _ = map_rows_to_template(doc_id, extracted_rows, template_by_doc[doc_id])
        for row in submission_rows:
            votes_by_id[row['id']] = coerce_votes(row.get('votes', '0'))

        if index % 10 == 0 or index == len(active_docs):
            partial_rows = materialize_submission_rows(votes_by_id, scaffold_rows)
            validate_submission_rows(partial_rows, expected_ids)
            write_submission_rows(partial_rows, PARTIAL_OUTPUT_PATH)

    if missing_labels and not ALLOW_MISSING_LABELS:
        preview = ', '.join(missing_labels[:5])
        raise FileNotFoundError(f'Missing page-label files for {len(missing_labels)} pages. Examples: {preview}')

    final_rows = materialize_submission_rows(votes_by_id, scaffold_rows)
    validate_submission_rows(final_rows, expected_ids)
    write_submission_rows(final_rows, OUTPUT_PATH)
    return final_rows


if RUN_BUILD_SUBMISSION:
    submission_rows = build_submission_from_page_labels()
    print(f'Wrote {len(submission_rows)} rows to {OUTPUT_PATH}')
else:
    print(f'Set RUN_BUILD_SUBMISSION = True to build {OUTPUT_PATH.name} from one-sheet JSON labels in {LABEL_DIR}')


NameError: name 'doc_ids' is not defined

## Notes

- One image page produces one JSON label file in `data/labels`.
- EasyOCR now runs only on detected table regions instead of the whole page.
- If a page has no detected table region, its label is still written but `results` is left empty.
- Small tables are handled more conservatively by permissive table-region detection and a bottom-page fallback for early pages.
- Vote values stored in labels and submission are normalized to Arabic numerals only.
- `submission.csv` is built later by grouping page labels back to document level.
